# 00 · Project Overview

**Competition:** [RSNA Intracranial Hemorrhage Detection](https://www.kaggle.com/competitions/rsna-intracranial-hemorrhage-detection/overview) (Kaggle, 2019)

This project rebuilds a solution to the challenge from scratch and scores it with late submissions. The notebooks in this folder are meant to be read in order and cover the full pipeline, from data exploration to the final submission.

## The competition

- **Host:** Radiological Society of North America (RSNA), with the American Society of Neuroradiology (ASNR) and MD.ai.
- **Dates:** September 18 – November 13, 2019, in two stages. Stage 2 released a new test set in the last nine days.
- **Scale:** 1,345 teams, 432 of them on the final (stage-2) leaderboard; $25,000 in prizes.

Intracranial hemorrhage, bleeding inside the skull, is an emergency. Treatment depends on whether there is a bleed, where it is and what type it is, and it has to start fast. The diagnosis is made on a head CT, which a specialist reads slice by slice. The challenge asked for an algorithm that detects acute hemorrhage and its subtypes on these scans.

## The task

The input is one slice of a head CT: an axial image of 512 × 512 pixels, stored as DICOM, whose values convert to Hounsfield units. For every slice the model returns the probability of six labels:

| Label | Where the blood is |
|---|---|
| `epidural` | between the skull and the dura, the outer membrane around the brain |
| `subdural` | between the dura and the arachnoid, the middle membrane |
| `subarachnoid` | under the arachnoid, in the fluid spaces along the brain surface |
| `intraventricular` | inside the ventricles, the fluid-filled cavities of the brain |
| `intraparenchymal` | inside the brain tissue itself |
| `any` | any of the five; in the labels it equals their maximum |

- A slice can have **several subtypes at once**. Each label is its own yes/no question, not a choice of one class.
- The labels are per slice, but **the slices come from whole scans**. Each scan is a stack of 20–60 slices, about 35 on average. The DICOM metadata (patient, study and series IDs, slice position) rebuilds the stacks, and the organisers allowed its use.

So this is a classification problem on single slices, but it hides a 3D one: a bleed usually spans several neighbouring slices, and the slices above and below say a lot about the one in between. Every top solution used this context.

## Evaluation metric

Submissions are scored with a **weighted multi-label log loss**. For image $i$ and label $j$, with true value $y_{ij} \in \{0, 1\}$ and predicted probability $p_{ij}$:

$$\text{score} = \frac{1}{N} \sum_{i=1}^{N} \frac{\sum_j w_j \, \ell(y_{ij}, p_{ij})}{\sum_j w_j}, \qquad \ell(y, p) = -\big[\, y \log p + (1 - y) \log (1 - p) \,\big]$$

with $w = 2$ for `any` and $w = 1$ for each subtype, so $\sum_j w_j = 7$. Predictions are clipped to $[10^{-15},\ 1 - 10^{-15}]$. Lower is better.

- The weights were never officially published; the host only said that `any` weighs more. Probing of the stage-1 leaderboard and all top solutions agree on 2.
- **`any` alone is 2/7 of the score.**
- **Log loss scores probabilities, not rankings.** A confident miss is expensive: a positive label predicted at 0.001 costs $-\log 0.001 \approx 6.9$, against a winning score of 0.0438. The predictions have to be calibrated, not only well ordered.
- For scale: predicting the base rate of each label for every image scores about 0.22.

The submission has one row per image and label (`ID_<image>_<label>,Label`): 121,232 × 6 = 727,392 rows.

## Data

| Split | Images | Scans | Labels |
|-------|-------:|------:|--------|
| Train | 752,803 | 21,744 | 6 labels per image (`stage_2_train.csv`) |
| Test  | 121,232 |  3,518 | None (predictions are scored on Kaggle) |

- Train is the stage-1 training set plus the stage-1 test set, whose labels were released when stage 2 started.
- About 25,000 non-contrast head CT exams from three institutions (Stanford University, Thomas Jefferson University Hospital and the Federal University of São Paulo), labelled by 60 neuroradiologists from the ASNR. Slices are 3–5 mm thick.
- **Bleeds are common per scan but rare per slice:** 41% of training scans contain one, but only 14.3% of training images. By subtype: subdural 6.3%, intraparenchymal 4.8%, subarachnoid 4.7%, intraventricular 3.5%, epidural 0.4%.
- The data is not included in this repository; see the README for how to download it.

**Two labelling styles.**
- Each training scan was read by one neuroradiologist. RSNA's dataset description: "Single annotation for training and validation data."
- Each test scan was read by three, blinded to each other. The label is their 2-of-3 majority, and 525 scans with conflicts were settled by a senior neuroradiologist.
- The test set also has fewer bleeds: 13.1% of images instead of 14.3%, and epidural 0.17% instead of 0.42%.

No part of the training data was labelled the test way, so there is no local copy of the test labelling. Our validation scores rank models; how the test labelling changes the score can only be measured with late submissions (notebooks 03, 05 and 06).